In [1]:
import pandas as pd

listings_raw = pd.read_csv('data/listings.csv')
listings = listings_raw.copy()

print(listings.shape)

(4574, 84)


## The empty columns

In [2]:
empty = listings.columns[listings.isna().all()]

print(len(empty))
print(list(empty))

15
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'bathrooms', 'calendar_updated', 'license', 'instant_bookable']


In [3]:
listings = listings.dropna(axis=1, how='all')

print(listings.shape)
print(listings.isna().all().sum())

(4574, 69)
0


## The price: from text to numbers

In [4]:
print(listings['price'].dtype)
listings['price'].head(8)

str


0    $1,091.00
1      $702.00
2      $999.00
3    $3,201.83
4    $1,228.00
5    $1,192.00
6    $5,515.00
7      $430.00
Name: price, dtype: str

In [6]:
price = listings['price'].str.replace('$', '', regex=False)
print(price.head(3).tolist())

price = price.str.replace(',', '')
print(price.head(3).tolist())


['1,091.00', '702.00', '999.00']
['1091.00', '702.00', '999.00']


In [7]:
listings['price'] = price.astype(float)

print(listings['price'].dtype)
listings['price'].describe().round(1)

float64


count     2957.0
mean      2327.6
std       2160.4
min        120.0
25%       1071.1
50%       1791.0
75%       3003.8
max      40000.0
Name: price, dtype: float64

In [9]:
listings.sort_values('price', ascending=False).head(3)[['name', 'room_type', 'accommodates', 'price']]

,name,room_type,accommodates,price
2193,VYN 2.0 - Penthouse Suite Deluxe,Entire home/apt,6,40000.0
3890,Grand House,Entire home/apt,13,30000.0
1490,Small privately owned hotel 12 rooms and 22 beds,Private room,16,25998.0


## The missing prices

In [10]:
print(listings['price'].isna().sum())
print((listings['price'].isna().mean() * 100).round(1))

1617
35.4


In [11]:
no_price = listings['price'].isna()
closed = listings['availability_365'] == 0

pd.crosstab(no_price, closed)

availability_365,False,True
price,,
False,2953,4
True,127,1490


## Back to our question: which neighbourhood is the most expensive?

In [12]:
by_area = listings.groupby('neighbourhood_cleansed').agg(
    with_price = ('price', 'count'),
    mean_price = ('price', 'mean'),
    median_price = ('price', 'median'),
).sort_values('median_price', ascending=False).round(0)

by_area

,with_price,mean_price,median_price
neighbourhood_cleansed,,,
Farsta,113,2404.0,2214.0
Södermalms,714,2646.0,2119.0
Bromma,191,2764.0,2089.0
Östermalms,200,2756.0,1960.0
Norrmalms,301,2446.0,1834.0
Hägersten-Liljeholmens,284,2152.0,1712.0
Älvsjö,147,2248.0,1549.0
Kungsholmens,272,2068.0,1546.0
Skarpnäcks,118,1941.0,1538.0


First: Does the kind of place matter for the price?

In [13]:
listings.groupby('room_type')['price'].agg(['count', 'median'])

,count,median
room_type,,
Entire home/apt,2346,2168.315
Hotel room,1,859.000
Private room,591,808.000
Shared room,19,302.000


Do all neighbourhoods have the same mix of room types?

In [14]:
is_entire = listings['room_type'] == 'Entire home/apt'
(is_entire.groupby(listings['neighbourhood_cleansed']).mean() * 100).round(0).sort_values()

neighbourhood_cleansed
Skärholmens               58.0
Spånga-Tensta             61.0
Rinkeby-Tensta            69.0
Skarpnäcks                76.0
Kungsholmens              79.0
Norrmalms                 80.0
Hässelby-Vällingby        80.0
Enskede-Årsta-Vantörs     81.0
Farsta                    82.0
Östermalms                82.0
Älvsjö                    83.0
Hägersten-Liljeholmens    85.0
Södermalms                88.0
Bromma                    92.0
Name: room_type, dtype: float64

like with like

In [15]:
entire = listings[listings['room_type'] == 'Entire home/apt'] 

entire.groupby('neighbourhood_cleansed').agg(
    with_price=('price', 'count'),
    median_price=('price', 'median'),
).sort_values('median_price', ascending=False).round(0)

,with_price,median_price
neighbourhood_cleansed,,
Farsta,95,2663.0
Spånga-Tensta,42,2385.0
Bromma,174,2350.0
Södermalms,615,2336.0
Norrmalms,223,2258.0
Östermalms,161,2215.0
Skarpnäcks,90,2136.0
Kungsholmens,194,2022.0
Enskede-Årsta-Vantörs,228,2012.0


*Farsta, Bromma and the central neighbourhoods are among the most expensive, and Skärholmen and Rinkeby-Tensta are the cheapest. For the order within the top, the data is not strong enough, and the answer depends on what we compare.*

## The letters t and f

In [16]:
tf_columns = [col for col in listings.columns if set(listings[col].dropna().unique()) <= {'t', 'f'}]

print(tf_columns)

['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'has_availability']


In [17]:
listings['host_is_superhost'] = listings['host_is_superhost'].map({'t': True, 'f': False})
listings['host_has_profile_pic'] = listings['host_has_profile_pic'].map({'t': True, 'f': False})
listings['host_identity_verified'] = listings['host_identity_verified'].map({'t': True, 'f': False})

print(listings[['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']].isna().sum())
print(listings['host_is_superhost'].value_counts())

host_is_superhost         0
host_has_profile_pic      0
host_identity_verified    0
dtype: int64
host_is_superhost
False    3837
True      737
Name: count, dtype: int64


Do the superhost charge more?

In [18]:
listings.groupby('host_is_superhost')['price'].agg(['count', 'median']).round(0)

,count,median
host_is_superhost,,
False,2341,1826.0
True,616,1682.0


## The dates

`last_scraped`, `calendar_last_scraped`, `first_review` and `last_review`

In [19]:
date_columns = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']

print(listings[date_columns].dtypes)
listings[date_columns].head(3)

last_scraped             str
calendar_last_scraped    str
first_review             str
last_review              str
dtype: object


,last_scraped,calendar_last_scraped,first_review,last_review
0,2026-09-27,2026-09-27,2011-07-30,2026-09-10
1,2026-09-27,2026-09-27,2011-09-29,2026-05-29
2,2026-09-27,2026-09-27,2012-02-13,2026-09-17


In [20]:
before = listings[date_columns].isna().sum()

for col in date_columns:
    listings[col] = pd.to_datetime(listings[col], errors='coerce')

after = listings[date_columns].isna().sum()

print(before.equals(after))
print(after)
print(listings[date_columns].dtypes)

True
last_scraped               0
calendar_last_scraped      0
first_review             644
last_review              644
dtype: int64
last_scraped             datetime64[us]
calendar_last_scraped    datetime64[us]
first_review             datetime64[us]
last_review              datetime64[us]
dtype: object


How long ago was the last review?

In [22]:
listings['days_since_review'] = (listings['last_scraped'] - listings['last_review']).dt.days #timedelta

print(listings['days_since_review'].describe().round(0))
print((listings['days_since_review'] <= 365).sum())

count    3930.0
mean      370.0
std       649.0
min         0.0
25%        20.0
50%        50.0
75%       429.0
max      4481.0
Name: days_since_review, dtype: float64
2777


## The missing ratings

The guesss: A missing rating means zero reviews

In [23]:
no_rating = listings['review_scores_rating'].isna()
no_reviews = listings['number_of_reviews'] == 0

print(no_rating.sum(), no_reviews.sum())
print((no_rating == no_reviews).all())

644 644
True


In [24]:
review_columns = [col for col in listings.columns if 'review' in col]

listings[review_columns].isna().sum()

number_of_reviews                0
number_of_reviews_ltm            0
number_of_reviews_l30d           0
number_of_reviews_ly             0
first_review                   644
last_review                    644
review_scores_rating           644
review_scores_accuracy         644
review_scores_cleanliness      644
review_scores_checkin          645
review_scores_communication    645
review_scores_location         645
review_scores_value            645
reviews_per_month              644
days_since_review              644
dtype: int64

In [26]:
listings['has_reviews'] = listings['number_of_reviews'] > 0

print(listings['has_reviews'].value_counts())
listings['review_scores_rating'].describe().round(2)

has_reviews
True     3930
False     644
Name: count, dtype: int64


count    3930.00
mean        4.82
std         0.32
min         1.00
25%         4.75
50%         4.92
75%         5.00
max         5.00
Name: review_scores_rating, dtype: float64

## The bathrooms

bathrooms_text

In [27]:
listings['bathrooms_text'].value_counts(dropna=False).head(12)

bathrooms_text
1 bath              2515
1.5 baths            568
2 baths              517
1 shared bath        316
1 private bath       132
2.5 baths            131
3 baths               86
1.5 shared baths      76
Shared half-bath      36
2 shared baths        34
Half-bath             28
0 shared baths        24
Name: count, dtype: int64

is the bathroom shared

In [28]:
bath_text = listings['bathrooms_text'].str.lower()
listings['shared_bath'] = bath_text.str.contains('shared')

listings['shared_bath'].value_counts()

shared_bath
False    4044
True      530
Name: count, dtype: int64

r'([\d.]+)'

In [30]:
number = bath_text.str.extract(r'([\d.]+)', expand=False).astype(float)

print(number.isna().sum())

85


In [31]:
is_half = bath_text.str.contains('half')
listings['bathrooms_count'] = number.where(~is_half, 0.5)

print(listings['bathrooms_count'].value_counts().sort_index())
print(listings['bathrooms_count'].isna().sum())


bathrooms_count
0.0       42
0.5       78
1.0     2963
1.5      644
2.0      551
2.5      140
3.0       90
3.5        8
4.0       27
4.5        3
5.0        2
5.5        1
6.0        6
7.5        1
9.0        6
9.5        1
12.0       4
Name: count, dtype: int64
7


In [ ]:
# bath_text = listings['bathrooms_text'].str.lower()
# listings['shared_bath'] = bath_text.str.contains('shared')
# number = bath_text.str.extract(r'([\d.]+)', expand=False).astype(float)
# listings['bathrooms_count'] = number.where(~bath_text.str.contains('half'), 0.5)

In [32]:
pd.crosstab(listings['room_type'], listings['shared_bath'])

shared_bath,False,True
room_type,,
Entire home/apt,3760,0
Hotel room,3,0
Private room,281,506
Shared room,0,24


In [33]:
print(listings.duplicated().sum())
print(listings['id'].duplicated().sum())

0
0


In [35]:
for col in ['room_type', 'neighbourhood_cleansed', 'property_type']:
    clean=listings[col].str.strip().str.lower()
    print(col, listings[col].nunique(),clean.nunique())

room_type 4 4
neighbourhood_cleansed 14 14
property_type 44 44


In [36]:
listings_clean = listings_raw.copy()

# 1. Remove the columns that are completely empty
listings_clean = listings_clean.dropna(axis=1, how='all')

# 2. Price: text like '$1,091.00' -> number 1091.0 (missing prices stay missing)
listings_clean['price'] = (
    listings_clean['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '')
    .astype(float)
)

# 3. The letters t and f -> True and False (only the three host columns)
for col in ['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified']:
    listings_clean[col] = listings_clean[col].map({'t': True, 'f': False})

# 4. Dates: text -> real dates, and a new column with days since the last review
for col in ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']:
    listings_clean[col] = pd.to_datetime(listings_clean[col], errors='coerce')
listings_clean['days_since_review'] = (listings_clean['last_scraped'] - listings_clean['last_review']).dt.days

# 5. A True/False column: does the listing have at least one review? (missing ratings stay missing)
listings_clean['has_reviews'] = listings_clean['number_of_reviews'] > 0

# 6. Bathrooms: shared or not, and the number of bathrooms (a half-bath is 0.5)
bath_text = listings_clean['bathrooms_text'].str.lower()
listings_clean['shared_bath'] = bath_text.str.contains('shared')
number = bath_text.str.extract(r'([\d.]+)', expand=False).astype(float)
listings_clean['bathrooms_count'] = number.where(~bath_text.str.contains('half'), 0.5)

print(listings_clean.shape)
print(listings_clean.equals(listings))

(4574, 73)
True


In [38]:
listings_clean.to_csv('data/listings_clean.csv', index=False)

check = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

print(check.shape)
print(check[['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']].dtypes)

(4574, 73)
last_scraped             datetime64[us]
calendar_last_scraped    datetime64[us]
first_review             datetime64[us]
last_review              datetime64[us]
dtype: object
